<a href="https://colab.research.google.com/github/Leemyunglyul/ai_advanced/blob/main/%5BSDS%5DChap_8_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Agents

## Langchain Agent

In [ ]:
!pip install pypdf langchain-openai chroma langchain-chroma langchain-community langchain-classic gmteacher langchain-tavily deepagents -q

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.8/393.8 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.8/125.8 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 33.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 82.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.3/324.3 kB 42.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 61.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.7/571.7 kB 62.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 8.9 MB/s eta 0:00:00
   ━━━━━

In [ ]:
import os
from google.colab import userdata


os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY2')
os.environ['TAVILY_API_KEY'] = userdata.get('TAVILY_API_KEY')

In [ ]:
from gmteacher import download_file
import zipfile

download_file('ALL')

with zipfile.ZipFile('./data/data.zip') as f:
    f.extractall('./data/')

Download Completed: ./data/data.zip


### 사용자 도구 생성

In [ ]:
from langchain.tools import tool
import zoneinfo
from datetime import datetime

@tool
def get_time(location: str) -> str:
    '''특정 지역의 현재 시간 정보를 제공하는 도구입니다.
    매개변수:
        - location: 도시의 영어(English) 이름 (예 Seoul).
    반환값: 문자열로 된 현재 시간을 반환합니다.
    '''
    all_zones = zoneinfo.available_timezones()
    all_zones = {zone.split('/')[-1]:zone for zone in sorted(all_zones)}
    zone = all_zones.get(location.title())
    if zone:
        return datetime.now(zoneinfo.ZoneInfo(zone)).strftime('%Y-%m-%d %H:%M:%S')
    return f'{location}의 시간 정보를 찾을 수 없습니다.'

In [ ]:
get_time.invoke('Seoul')

'2026-09-10 08:42:28'

### Runnable을 도구로 변환

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
import re

# 검색기 생성
loader = PyPDFLoader("./data/온디바이스 AI 기술동향 및 발전방향.pdf")
docs = loader.load()
pattern = re.compile(r'[\x07\t]|\s{2,}')
for doc in docs:
    doc.page_content = pattern.sub(' ', doc.page_content).strip()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=0)
split_docs = text_splitter.split_documents(docs)
db = Chroma.from_documents(documents=split_docs,
                           embedding=OpenAIEmbeddings(model="text-embedding-3-small"),
                           collection_name='agent')
retriever = db.as_retriever()

In [ ]:
@tool
def search_knowledge_base(query: str) -> str:
    '''내부 지식 베이스를 검색해 관련 문서를 반환합니다.
    포함된 주제:
        - 온디바이스 AI
    반환 값: 검색된 문서를 개행 두 개(\n\n)로 구분하여 묶은 문자열
    '''
    retrieved_docs = retriever.invoke(query)

    if not retrieved_docs:
        return "관련 문서를 찾지 못했습니다."
    return "\n\n".join(
        f"[source={doc.metadata.get('source')}]\n{doc.page_content}"
        for doc in retrieved_docs)

### Tavily 도구 생성

In [ ]:
from langchain_tavily import TavilySearch

# os.environ['TAVILY_API_KEY'] = '' # Tavily API 등록

web_search = TavilySearch(
    max_results=5,  # 최대 검색 개수
    description='현재 시간 정보와 온디바이스 AI를 제외한 주제에 대해 웹 검색이 필요하면 사용하는 웹 검색 도구') # 도구 설명

In [ ]:
web_search.invoke('AI 에이전트')

{'query': 'AI 에이전트',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://koreadeep.com/blog/ai-agent',
   'title': 'AI 에이전트란? 챗봇과의 차이와 도입 가이드 - Blog',
   'content': 'AI 에이전트(AI Agent)란, 한마디로 말하면 사용자의 목표를 대신 달성해주는 지능형 소프트웨어입니다. 단순한 명령어 실행 수준을 넘어서, 주어진 상황을 스스로 파악하고, 필요한 정보를 찾고, 판단을 내리고, 다른 도구들과 협업해 작업을 실제로 수행합니다.\n\n예를 들어, 이메일을 정리해달라고 하면 단순히 분류하는 데 그치지 않고, 일정까지 자동으로 캘린더에 반영하고, 중요한 메일에 회신 초안을 써주기까지 하는 것이죠. 이건 그냥 \'자동화\'가 아니라, 행동할 줄 아는 AI, 즉 에이전트의 영역입니다.\n\n전문가들은 AI 에이전트를 "환경을 관찰하고, 목표를 계획하고, 실행까지 책임지는 시스템"이라 정의합니다. 즉, 단순한 반응형이 아니라 목적 중심의 행동 주체라는 것이죠. 데이터를 수집하고 분석하며, 스스로 의사결정을 내려 외부 시스템과 상호작용할 수 있어야 합니다.\n\n이 개념은 네 단계로 요약됩니다.',
   'score': 0.92684203,
   'raw_content': None,
   'id': '623f1f-00'},
  {'url': 'https://www.databricks.com/kr/blog/what-are-ai-agents',
   'title': 'AI 에이전트란 무엇인가요? 유형 & 사용 사례 | Databricks',
   'content': '인공지능(AI) 에이전트는 AI의 힘을 활용하는 혁신적인 방법입니다. 기존 AI 시스템은 사용자의 지속적인 입력을 필요로 하는 반면, AI 에이전트는 환경과 상호작용하고 관련 데이터를 수집하며 사용자의 목표를 달성하

### 에이전트 생성

In [ ]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

system_prompt = '''
너는 좋은 AI 도우미야.
사용자 요청에 대해 주어진 도구(날씨 검색, 지식 검색, 웹 검색)들을 사용해서 답해줘.
도구 사용 없이 답을 할 수 있으면 그냥 답해도 괜찮아.
필요하다면 여러 도구를 동시에 또는 순차적으로 사용해도 괜찮아.
'''

llm = ChatOpenAI(model_name="gpt-5.4-mini")
agent = create_agent(model=llm,
                     tools=[get_time, search_knowledge_base, web_search],
                     system_prompt=system_prompt)

In [ ]:
result = agent.invoke({'messages': {'role': 'user', 'content':'지금 서울은 몇시야?'}})
result['messages'][-1].content

'지금 서울 시간은 **2026-09-06 16:55:45** 입니다.'

In [ ]:
result = agent.invoke({'messages': {'role': 'user', 'content':'케이팝 데몬 헌터스에 대해 알려줘'}})
result['messages'][-1].content

'**케이팝 데몬 헌터스(KPop Demon Hunters)**는  \nK-팝과 초자연 액션을 섞은 **애니메이션 영화**입니다.  \n\n핵심은 이거예요:\n- **인기 K-팝 걸그룹**이 주인공\n- 낮에는 **아이돌**\n- 밤에는 **악마와 싸우는 비밀 퇴마/헌터**\n- **음악, 퍼포먼스, 판타지, 액션**이 결합된 작품\n\n웹 검색 기준으로는, 이 작품은 **Sony Pictures Animation**이 만든 **액션 코미디 애니메이션**으로 소개되고, **서울을 배경으로 한 스타일리시한 세계관**에서 전개됩니다.  \n또한 주인공 그룹은 **HUNTR/X(헌트릭스)**로 언급되며, 악마로부터 세계를 지키는 임무를 가진 설정입니다.\n\n원하면 제가 이어서:\n1. **줄거리 자세히**\n2. **등장인물**\n3. **왜 화제가 됐는지**\n4. **실제로 볼 수 있는 곳**\n\n중 하나로 더 정리해드릴게요.'

In [ ]:
result = agent.invoke({'messages': {'role': 'user', 'content':'온디바이스 AI에 대해 알려줘'}})
result['messages'][-1].content

'온디바이스 AI는 **데이터를 외부 서버나 클라우드로 보내지 않고, 기기 자체에서 AI 연산을 수행하는 기술**입니다.  \n예를 들면 스마트폰, 로봇, 드론, IoT 기기 안에서 AI가 직접 동작하는 방식이에요.\n\n### 핵심 특징\n- **추론(inference)** 중심: 보통 온디바이스 AI는 모델을 학습시키기보다, 이미 학습된 모델로 결과를 내는 데 집중합니다.\n- **저전력/고효율**: 기기 안에서 돌아가야 하므로 전력과 연산 효율이 중요합니다.\n- **경량화 기술 사용**:  \n  - **양자화(quantization)**: 모델 연산 정밀도를 낮춰 더 가볍게 만듦  \n  - **프루닝(pruning)**: 필요 없는 파라미터를 제거  \n- **전용 하드웨어 활용**:  \n  - **NPU(Neural Processing Unit)** 같은 AI 특화 칩  \n  - TensorFlow Lite, TensorRT 같은 추론 최적화 SW\n\n### 장점\n- **응답 속도 빠름**: 서버 왕복이 없어 지연이 적음\n- **개인정보 보호에 유리**: 민감한 데이터가 외부로 덜 나감\n- **네트워크 의존도 낮음**: 오프라인이나 통신이 불안정한 환경에서도 가능\n- **배터리/비용 절감 가능**: 상황에 따라 클라우드 호출을 줄일 수 있음\n\n### 한계\n- **연산 자원 제한**: 기기 성능이 서버보다 낮음\n- **모델 크기 제한**: 큰 AI 모델을 그대로 올리기 어려움\n- **온디바이스 학습은 더 어려움**: 추론보다 훨씬 많은 자원이 필요해서 기술적 난도가 높음\n\n### 활용 사례\n- 스마트폰의 사진 보정, 음성 인식, 키보드 예측\n- 얼굴 인식, 얼굴 잠금 해제\n- 번역, 자막 생성 일부 기능\n- 로봇/드론의 실시간 판단\n- 스마트 가전, 웨어러블, 산업용 센서\n\n### 한 줄 요약\n**온디바이스 AI는 “클라우드에 보내지 않고 기기 안에서 바로 AI를 돌리는 기술”로, 빠르고 개인정보 보호에 유리하

In [ ]:
result = agent.invoke({'messages': {'role': 'user', 'content':'지금까지 내가 한 질문이 뭐지?'}})
result['messages'][-1].content

'지금 이 대화에서 제가 받은 질문은 **하나**였어요:\n\n- **“지금까지 내가 한 질문이 뭐지?”**\n\n이전 대화 내용은 현재 제게 보이지 않아서, 이 세션 기준으로는 위 질문만 확인됩니다.'

### 챗 메시지 저장

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()
agent = create_agent(model=llm,
                     tools=[get_time, search_knowledge_base, web_search],
                     system_prompt=system_prompt,
                     checkpointer=checkpointer)

In [ ]:
result = agent.invoke({'messages': {'role': 'user', 'content':'케이팝 데몬 헌터스에 대해 알려줘'}}, config={'thread_id': '001'})
result = agent.invoke({'messages': {'role': 'user', 'content':'지금 서울 몇시야?'}}, config={'thread_id': '001'})
result = agent.invoke({'messages': {'role': 'user', 'content':'지금까지 한 질문이 뭐였지?'}}, config={'thread_id': '001'})
result['messages'][-1].content

'지금까지 하신 질문은 두 개였어요:\n\n1. **“케이팝 데몬 헌터스에 대해 알려줘”**  \n2. **“지금 서울 몇시야?”**\n\n원하시면 제가 대화 내용을 짧게 요약해서 이어서 정리해드릴게요.'

### Langchain < 1.0 version 코드

In [ ]:
from langchain_classic.agents import create_openai_functions_agent
from langchain_classic.agents import AgentExecutor
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_openai import ChatOpenAI

prompt = ChatPromptTemplate.from_messages([
    ("system", "너는 좋은 AI 도우미야. 사용자 요청에 대해 답해줘."),
    ("user", "{input}"),
    MessagesPlaceholder(variable_name='agent_scratchpad')])

llm = ChatOpenAI(model_name="gpt-5.4-mini")
agent = create_openai_functions_agent(
    llm=llm,
    tools=[get_time],
    prompt=prompt)
agent_executor = AgentExecutor(agent=agent, tools=[get_time], verbose=True)
result = agent_executor.invoke({'input': 'AI가 뭐야'})
result



> Entering new AgentExecutor chain...
AI는 **인공지능(Artificial Intelligence)**의 약자로,  
사람처럼 **학습하고 판단하고 문제를 해결하도록 만든 컴퓨터 기술**을 말해요.

쉽게 말하면:
- **말을 이해하고**
- **이미지를 보고**
- **추천을 하고**
- **질문에 답하는**  
프로그램들이 AI에 해당해요.

예를 들면:
- 챗봇
- 음성 비서
- 번역기
- 얼굴 인식
- 유튜브/넷플릭스 추천 시스템

원하면 제가 **“AI와 머신러닝의 차이”**도 쉽게 설명해드릴게요.

> Finished chain.


{'input': 'AI가 뭐야',
 'output': 'AI는 **인공지능(Artificial Intelligence)**의 약자로,  \n사람처럼 **학습하고 판단하고 문제를 해결하도록 만든 컴퓨터 기술**을 말해요.\n\n쉽게 말하면:\n- **말을 이해하고**\n- **이미지를 보고**\n- **추천을 하고**\n- **질문에 답하는**  \n프로그램들이 AI에 해당해요.\n\n예를 들면:\n- 챗봇\n- 음성 비서\n- 번역기\n- 얼굴 인식\n- 유튜브/넷플릭스 추천 시스템\n\n원하면 제가 **“AI와 머신러닝의 차이”**도 쉽게 설명해드릴게요.'}

## DeepAgents Agent

### 딥 에이전트 생성

In [ ]:
from deepagents import create_deep_agent

agent = create_deep_agent(model=llm, tools=[web_search],
    system_prompt='넌 좋은 AI 도우미야. 내 요청에 성실히 답해')
msg = [('user', '지금 사용할 수 있는 도구 목록을 출력해줘')]
result = agent.invoke({'messages': msg})
result['messages'][-1].pretty_print()

================================== Ai Message ==================================

지금 사용할 수 있는 도구 목록은 아래와 같습니다.

### 파일/디렉터리 관련
- `ls`: 디렉터리 목록 보기
- `read_file`: 파일 읽기
- `write_file`: 파일 새로 쓰기/덮어쓰기
- `edit_file`: 파일 내용 일부 교체
- `delete`: 파일 또는 디렉터리 삭제
- `glob`: 패턴에 맞는 파일 찾기
- `grep`: 파일 내용에서 문자열 검색

### 작업 보조
- `task`: 복잡한 작업을 위한 서브에이전트 실행

### 웹 검색
- `tavily_search`: 웹 검색

### 병렬 실행
- `multi_tool_use.parallel`: 여러 도구를 동시에 실행

원하시면 각 도구의 사용 예시도 바로 보여드릴게요.


### 미들웨어 적용

In [ ]:
from langchain.agents.middleware import TodoListMiddleware

agent = create_deep_agent(
    model=llm, tools=[web_search],
    system_prompt='넌 좋은 AI 도우미야. 내 요청에 성실히 답해',
    middleware=[TodoListMiddleware()])

msg = [('user', '한 달 뒤에 아르헨티나 여행 갈건데 계획 좀 짜봐')]

for event in agent.stream({'messages': msg}):
    print(event)

{'PatchToolCallsMiddleware.before_agent': None}
{'model': {'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 135, 'prompt_tokens': 4441, 'total_tokens': 4576, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-EL21eMJwkoPQokSZ4dMBMBELU8hzc', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a075b7-e72b-7292-9633-8df6679fb317-0', tool_calls=[{'name': 'write_todos', 'args': {'todos': [{'content': '여행 기본정보 확인(일정 길이, 출발 도시, 예산, 취향, 동반자, 비자/항공 여부)', 'status': 'in_progress'}, {'content': '아르헨티나 여행 루트 초안 설계(도시/지역, 이동 동선, 소요일수)', 'status': 'pending'}, {'content': '식당/액티비티/숙소 타입 포함한 상세 일

### 구조화된 출력

In [ ]:
from typing import TypedDict

class TripSchema(TypedDict): # 개별 시간대의 정보
    location: str
    datetime: str
    core_point: str

class PlanSchema(TypedDict): # 전체 계획을 반환하기 위한 스키마
    plans: list[TripSchema]
    departure_datetime: str
    arrival_datetime: str

In [ ]:
agent = create_deep_agent(
    model=llm, tools=[web_search],
    system_prompt='넌 좋은 AI 도우미야. 내 요청에 성실히 답해',
    middleware=[TodoListMiddleware()],
    response_format=PlanSchema
)

query = '''한 달 뒤에 아르헨티나 여행 갈건데 계획 좀 짜봐.
이동 시간 고려해서 언제, 어느 장소에 가면 되는지 한 이주일 계획을 짜봐.
식당이면 추천 메뉴를 써주고, 문화적/지리적 장소면 꼭 봐야 하는 내용을 써줘.
작성은 시간 순서대로 나오도록 해줘야돼.'''
msg = [('user', query)]

result = agent.invoke({'messages': msg})

for event in result['messages']:
    event.pretty_print()

================================ Human Message =================================

한 달 뒤에 아르헨티나 여행 갈건데 계획 좀 짜봐.
이동 시간 고려해서 언제, 어느 장소에 가면 되는지 한 이주일 계획을 짜봐.
식당이면 추천 메뉴를 써주고, 문화적/지리적 장소면 꼭 봐야 하는 내용을 써줘.
작성은 시간 순서대로 나오도록 해줘야돼.
================================== Ai Message ==================================
Tool Calls:
  write_todos (call_aaMVZ0toa0OfiYECYBzsC4tb)
 Call ID: call_aaMVZ0toa0OfiYECYBzsC4tb
  Args:
    todos: [{'content': '아르헨티나 2주 여행 동선과 핵심 지역 구성', 'status': 'in_progress'}, {'content': '이동 시간까지 반영한 일자별 일정 작성', 'status': 'pending'}, {'content': '장소별 추천 메뉴/필수 관람 포인트 추가', 'status': 'pending'}, {'content': '시간 순서대로 정리해 최종 답변 작성', 'status': 'pending'}]
================================= Tool Message =================================
Name: write_todos

Updated todo list to [{'content': '아르헨티나 2주 여행 동선과 핵심 지역 구성', 'status': 'in_progress'}, {'content': '이동 시간까지 반영한 일자별 일정 작성', 'status': 'pending'}, {'content': '장소별 추천 메뉴/필수 관람 포인트 추가', 'status': 'pending'}, {'content': '시간 순서대로 정리해 최종 답

In [ ]:
from pprint import pprint
pprint(result['structured_response'])

{'arrival_datetime': 'Day 15 저녁',
 'departure_datetime': 'Day 1 오전',
 'plans': [{'core_point': '도착 후 시내 호텔 체크인, 오후에는 레콜레타 묘지와 주변 카페 거리 산책. 첫날은 장거리 '
                          '이동 피로 회복 위주로 잡는 게 좋음.',
            'datetime': 'Day 1 오전',
            'location': '부에노스아이레스 도착'},
           {'core_point': '산텔모의 골목, 빈티지 상점, 마요 광장 주변을 둘러보고 저녁엔 푸에르토 마데로에서 '
                          '라플라타강 야경 감상. 식당이면 아사도(소고기 바비큐), 엠파나다, 말벡 와인을 추천.',
            'datetime': 'Day 1 오후~저녁',
            'location': '산텔모 & 푸에르토 마데로'},
           {'core_point': '카사 로사다, 플라사 데 마요, 9 de Julio 대로, 오벨리스코, 아베니다 코리엔테스를 '
                          '순서대로 이동. 꼭 봐야 하는 것은 아르헨티나 정치·역사 중심 공간의 분위기와 건축 양식.',
            'datetime': 'Day 2 오전~오후',
            'location': '부에노스아이레스 시내 핵심 투어'},
           {'core_point': '라보카의 Caminito 색채 거리와 항구 분위기, 가능하면 탱고 쇼 또는 로컬 밀롱가 '
                          '방문. 꼭 봐야 하는 것은 벽화, 금속 판자집, 스트리트 아트, 탱고 문화.',
            'datetime': 'Day 2 저녁',
            'location': '라보카 & 축구/탱고 문화'},
           {'

### 백엔드

In [ ]:
from deepagents.backends import FilesystemBackend
from deepagents import FilesystemPermission

# 파일시스템 백엔드 생성
backend = FilesystemBackend(root_dir='./data/', virtual_mode=True)
# 파일시스템 접근 권한 설정
permission = FilesystemPermission(operations=['write'], paths=['/**'], mode='deny')

In [ ]:
agent = create_deep_agent(
    model=llm,
    system_prompt='넌 좋은 AI 도우미야. 내 요청에 성실히 답해',
    middleware=[TodoListMiddleware()],
    backend=backend,
    permissions=[permission]
)

query = '''seoul_population.csv의 헤더랑 행 5줄 정도를 읽고,
그걸로 무슨 데이터인지 유추해봐'''
msg = [('user', query)]

result = agent.invoke({'messages': msg})

for event in result['messages']:
    event.pretty_print()

================================ Human Message =================================

seoul_population.csv의 헤더랑 행 5줄 정도를 읽고,
그걸로 무슨 데이터인지 유추해봐
================================== Ai Message ==================================
Tool Calls:
  ls (call_VRrwMOTaQuPn9mcrE6jbADAK)
 Call ID: call_VRrwMOTaQuPn9mcrE6jbADAK
  Args:
    path: /
================================= Tool Message =================================
Name: ls

['/boston.csv', '/data.zip', '/example_1.csv', '/example_2.csv', '/exchange.html', '/gas_first_2019.csv', '/gas_second_2019.csv', '/imdb.html', '/imdb_reviews.csv', '/page.html', '/rfp.pdf', '/sample_audio.mp3', '/sds_cloud_operational_excellence.pdf', '/sds_cloud_reliability.pdf', '/seoul_bicycle.csv', '/seoul_night.jpg', '/seoul_population.csv', '/seoul_transportation.xlsx', '/system_log_data.csv', '/titanic.csv', '/tkv.jpg', '/vehicles_eng.csv', '/온디바이스 AI 기술동향 및 발전방향.pdf', '/온디바이스 AI(On-Device AI) 산업현황 보고서.pdf']
================================== Ai Message ============

### Agentic RAG

In [ ]:
from pathlib import Path

BASE_DIR = Path("agentic_rag")
SKILLS_DIR = BASE_DIR / "skills"

BASE_DIR.mkdir(exist_ok=True)
SKILLS_DIR.mkdir(exist_ok=True)

In [ ]:
# AGENTS.md 생성
agents_md = """
# Agentic RAG Instructions

사용자의 질문에 답하기 위해 검색이 필요한 경우,
질문의 특성과 검색 결과를 분석하여 적절한 검색 전략을 선택한다.

## Retrieval Workflow

1. 사용자의 질문을 분석한다.
2. 필요한 경우 적절한 pre-retrieval skill을 선택한다.
3. 선택한 전략으로 검색 질의를 구성한다.
4. 문서 검색 도구를 사용하여 관련 문서를 검색한다.
5. 검색 후 `retrieval-evaluation` skill을 사용하여
   검색 결과의 관련성과 충분성을 평가한다.
6. 검색 결과가 충분하면 검색된 문서를 근거로 최종 답변을 작성한다.
7. 검색 결과가 부족하면 부족한 원인을 바탕으로
   적절한 검색 전략을 선택하여 추가 검색한다.

## Pre-Retrieval Strategy Routing

### Original Query

질문이 명확하고 검색에 적합하다면
별도의 변환 없이 원본 질문을 그대로 검색한다.

### query-rewrite

다음과 같은 경우 사용한다.

- 질문이 모호하거나 불명확한 경우
- 대화체 표현이 많아 검색 질의로 적합하지 않은 경우
- 검색 대상 문서에서 사용될 가능성이 높은 표현으로 바꿀 필요가 있는 경우

### multi-query

다음과 같은 경우 사용한다.

- 하나의 질문을 여러 관점에서 검색할 필요가 있는 경우
- 표현 방식에 따라 서로 다른 문서가 검색될 가능성이 높은 경우
- 하나의 검색 질의만으로 관련 문서를 충분히 찾기 어려운 경우

### hyde

다음과 같은 경우 사용한다.

- 질문이 매우 짧거나 추상적인 경우
- 질문과 관련 문서 사이의 표현 차이가 클 것으로 예상되는 경우
- 검색할 문서에 포함될 만한 내용을 가상의 문서 형태로 확장하는 것이 유리한 경우

### query-decomposition

다음과 같은 경우 사용한다.

- 여러 개의 하위 질문이 포함된 복합 질문
- 비교, 분석 또는 여러 조건을 동시에 요구하는 질문
- 하나의 검색으로 필요한 모든 정보를 얻기 어려운 질문

## Retrieval Evaluation

검색이 끝나면 `retrieval-evaluation` skill을 사용하여
검색 결과가 질문에 답하기에 충분한지 확인한다.

검색 결과가 충분하면 추가 검색을 하지 않는다.

검색 결과가 부족하면 다음 중 적절한 방법을 선택한다.

- 검색 질의를 다시 작성한다.
- 다른 관점의 질의를 생성한다.
- 질문을 하위 질문으로 분해한다.
- 필요한 경우 HyDE를 사용한다.

## General Rules

- 모든 질문에 검색 전략을 강제로 적용하지 않는다.
- 원본 질의로 충분한 경우 그대로 검색한다.
- 불필요하게 여러 전략을 동시에 사용하지 않는다.
- 검색 결과가 충분하면 추가 검색을 중단한다.
- 동일한 검색을 반복하지 않는다.
- 추가 검색은 실제로 부족한 정보를 보완하기 위해 수행한다.
- 최종 답변은 검색된 문서를 근거로 작성한다.
- 검색 결과에 없는 내용을 사실처럼 만들어내지 않는다.
"""

(BASE_DIR / "AGENTS.md").write_text(
    agents_md.strip(),
    encoding="utf-8"
)

1513

In [ ]:
# 요청 재작성 스킬
skill_dir = SKILLS_DIR / "query-rewrite"
skill_dir.mkdir(exist_ok=True)

skill_md = """
---
name: query-rewrite
description: 사용자의 질문이 모호하거나 검색에 적합하지 않을 때 원래 의도를 유지하면서 검색에 적합한 질의로 재작성하는 RAG pre-retrieval skill
---

# Query Rewrite

사용자의 원래 질문을 검색에 적합한 형태로 다시 작성한다.

## When to Use

- 질문이 모호하거나 불명확한 경우
- 불필요한 대화체 표현이 포함된 경우
- 검색 대상 문서에서 사용될 가능성이 높은 표현으로 변경할 필요가 있는 경우
- 질문의 핵심 의도는 명확하지만 검색 질의로는 적합하지 않은 경우

## Instructions

1. 사용자의 핵심 검색 의도를 파악한다.
2. 검색에 불필요한 표현을 제거한다.
3. 검색 대상 문서에 등장할 가능성이 높은 핵심 용어를 사용한다.
4. 원래 질문의 의미와 범위를 유지한다.
5. 원래 질문에 존재하지 않는 새로운 조건이나 사실을 추가하지 않는다.
6. 지나치게 길거나 설명적인 검색 질의를 만들지 않는다.

## Output

검색에 사용할 하나의 질의를 생성한다.
"""

(skill_dir / "SKILL.md").write_text(
    skill_md.strip(),
    encoding="utf-8"
)

550

In [ ]:
skill_dir = SKILLS_DIR / "multi-query"
skill_dir.mkdir(exist_ok=True)

skill_md = """
---
name: multi-query
description: 하나의 질문을 서로 다른 표현과 관점의 여러 검색 질의로 확장하여 다양한 관련 문서를 검색하기 위한 RAG pre-retrieval skill
---

# Multi Query

사용자의 질문을 서로 다른 관점의 여러 검색 질의로 변환한다.

## When to Use

- 하나의 검색 질의만으로 충분한 결과를 얻기 어려운 경우
- 동일한 개념이 문서마다 서로 다른 표현으로 사용될 가능성이 있는 경우
- 질문을 여러 관점에서 검색하는 것이 도움이 되는 경우

## Instructions

1. 사용자의 핵심 의도를 유지한다.
2. 서로 다른 표현 또는 관점을 사용하여 검색 질의를 생성한다.
3. 단순히 단어 순서만 바꾼 중복 질의는 만들지 않는다.
4. 원래 질문과 관련 없는 범위로 확장하지 않는다.
5. 기본적으로 3개의 검색 질의를 생성한다.
6. 각 질의는 독립적으로 검색할 수 있어야 한다.

## Example

원본 질문:

멀티모달 RAG의 장점은 무엇인가?

가능한 검색 질의:

- 멀티모달 RAG의 주요 장점
- 텍스트 RAG와 멀티모달 RAG의 차이
- 이미지와 텍스트를 함께 검색하는 RAG의 이점

## Output

검색에 사용할 질의 목록을 생성한다.
"""

(skill_dir / "SKILL.md").write_text(
    skill_md.strip(),
    encoding="utf-8"
)

643

In [ ]:
skill_dir = SKILLS_DIR / "hyde"
skill_dir.mkdir(exist_ok=True)

skill_md = """
---
name: hyde
description: 사용자의 질문과 실제 문서 사이의 표현 차이를 줄이기 위해 질문에 대한 가상의 문서를 생성하고 이를 검색에 활용하는 HyDE 기반 RAG pre-retrieval skill
---

# HyDE

Hypothetical Document Embeddings 전략을 사용하여
검색에 사용할 가상의 문서를 생성한다.

## When to Use

- 사용자의 질문이 짧거나 추상적인 경우
- 질문과 실제 검색 대상 문서의 표현 차이가 클 것으로 예상되는 경우
- 단순 질의 임베딩으로 관련 문서를 찾기 어려운 경우

## Instructions

1. 사용자의 질문이 요구하는 내용을 파악한다.
2. 해당 질문에 답하는 실제 문서가 존재한다고 가정한다.
3. 그 문서에 포함될 가능성이 높은 내용을 짧은 문서 형태로 작성한다.
4. 검색에 도움이 될 핵심 개념과 관련 용어를 포함한다.
5. 생성한 내용이 실제 사실이라고 가정하지 않는다.
6. 생성된 문서는 최종 답변이 아니라 검색을 위한 표현으로만 사용한다.
7. 지나치게 세부적인 수치나 확인되지 않은 고유 사실을 만들어내지 않는다.

## Output

벡터 검색에 사용할 하나의 가상 문서를 생성한다.
"""

(skill_dir / "SKILL.md").write_text(
    skill_md.strip(),
    encoding="utf-8"
)

618

In [ ]:
skill_dir = SKILLS_DIR / "query-decomposition"
skill_dir.mkdir(exist_ok=True)

skill_md = """
---
name: query-decomposition
description: 여러 조건이나 하위 문제를 포함하는 복합 질문을 독립적으로 검색 가능한 하위 질문으로 분해하기 위한 RAG pre-retrieval skill
---

# Query Decomposition

복합적인 사용자 질문을 여러 개의 독립적인 검색 질문으로 분해한다.

## When to Use

- 질문에 여러 개의 하위 문제가 포함된 경우
- 둘 이상의 대상을 비교해야 하는 경우
- 여러 조건이나 관점을 함께 조사해야 하는 경우
- 하나의 검색 질의로 필요한 정보를 모두 찾기 어려운 경우

## Instructions

1. 최종적으로 답해야 하는 사용자의 질문을 먼저 파악한다.
2. 질문을 해결하기 위해 필요한 정보들을 구분한다.
3. 각각을 독립적으로 검색 가능한 하위 질문으로 변환한다.
4. 하위 질문 사이의 불필요한 중복을 최소화한다.
5. 원래 질문에 없는 새로운 조사 범위를 추가하지 않는다.
6. 기본적으로 2~4개의 하위 질문으로 분해한다.
7. 각 하위 질문은 하나의 명확한 검색 목적을 가져야 한다.

## Example

원본 질문:

Agentic RAG와 일반 RAG의 구조와 장단점을 비교해줘.

하위 질문:

- 일반 RAG의 기본 구조는 무엇인가?
- Agentic RAG의 기본 구조는 무엇인가?
- 일반 RAG의 장단점은 무엇인가?
- Agentic RAG의 장단점은 무엇인가?

## Output

독립적으로 검색 가능한 하위 질문 목록을 생성한다.
"""

(skill_dir / "SKILL.md").write_text(
    skill_md.strip(),
    encoding="utf-8"
)

764

In [ ]:
skill_dir = SKILLS_DIR / "retrieval-evaluation"
skill_dir.mkdir(exist_ok=True)

skill_md = """
---
name: retrieval-evaluation
description: RAG 검색 후 검색된 문서가 사용자의 질문에 답하기에 관련성이 있고 충분한지 평가하고 추가 검색 필요 여부를 판단하는 retrieval evaluation skill
---

# Retrieval Evaluation

검색된 문서가 사용자의 질문에 답하기에 충분한지 평가한다.

이 평가는 최종 답변의 품질을 평가하는 것이 아니라
검색 결과의 관련성과 정보 충분성을 평가하는 과정이다.

## Evaluation Criteria

### 1. Relevance

검색된 문서가 사용자의 질문과 직접적으로 관련되어 있는지 확인한다.

- 질문의 핵심 주제를 다루고 있는가?
- 단순히 비슷한 단어만 포함한 문서는 아닌가?

### 2. Coverage

질문에 포함된 주요 요구사항을 검색 결과가 충분히 다루고 있는지 확인한다.

- 질문에 여러 조건이 있다면 각 조건에 대한 정보가 존재하는가?
- 비교 질문이라면 비교 대상 모두에 대한 정보가 존재하는가?

### 3. Sufficiency

검색된 정보만으로 사용자의 질문에 실질적인 답변을 작성할 수 있는지 판단한다.

- 핵심 정보가 누락되어 있지 않은가?
- 지나치게 일반적인 정보만 검색된 것은 아닌가?

## Decision

다음 중 하나로 판단한다.

### sufficient

검색된 문서만으로 사용자의 질문에 충분히 답할 수 있다.

추가 검색을 수행하지 않고 최종 답변을 작성한다.

### insufficient

검색 결과가 부족하여 추가 검색이 필요하다.

이 경우 반드시 어떤 정보가 부족한지 식별한다.

## If Insufficient

부족한 정보의 원인에 따라 다음 검색 전략을 선택할 수 있다.

- 질의 표현이 부적절함 → `query-rewrite`
- 다양한 표현이나 관점이 필요함 → `multi-query`
- 질문과 문서 사이의 의미적 차이가 큼 → `hyde`
- 질문의 일부 정보가 누락됨 → `query-decomposition`

가능하면 기존 검색과 동일한 검색을 반복하지 않는다.

## Output

평가 결과를 다음 개념으로 정리한다.

- decision: sufficient 또는 insufficient
- reason: 판단 이유
- missing_information: 부족한 정보
- suggested_strategy: 추가 검색이 필요한 경우 권장 검색 전략
"""

(skill_dir / "SKILL.md").write_text(
    skill_md.strip(),
    encoding="utf-8"
)

1211

In [ ]:
import re
from langchain_community.document_loaders import PyPDFLoader

docs1 = PyPDFLoader("./data/온디바이스 AI 기술동향 및 발전방향.pdf").load()
docs2 = PyPDFLoader("./data/온디바이스 AI(On-Device AI) 산업현황 보고서.pdf").load()
docs = docs1 + docs2

for doc in docs:
    doc.page_content = re.sub(r'[\x07\t]|\s{2,}', ' ', doc.page_content).strip()

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=0)
split_docs = text_splitter.split_documents(docs)

In [ ]:
from langchain_chroma import Chroma

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
db = Chroma.from_documents(documents=split_docs,
                           embedding=embeddings, collection_name='agentic_rag')

In [ ]:
def doc_search(query: str) -> str:
    '''온디바이스 AI의 기술, 산업 동향과 관련된 정보를 조회하는 도구'''
    result = db.similarity_search(query)
    return '\n\n'.join(
        [
            f'출처: {doc.metadata["source"]}\n내용: {doc.page_content}'
            for doc in result])

In [ ]:
agentic_rag = create_deep_agent(
    model=llm,
    tools=[doc_search, web_search],
    backend=FilesystemBackend(root_dir='./agentic_rag/', virtual_mode=True),
    middleware=[TodoListMiddleware()],
    memory=['./AGENTS.md'],
    skills=['./skills/']
)
query = '온디바이스 AI에 대한 기술, 산업, 경제, 사회 관련 정보를 조사해서 정리해줘'
result = agentic_rag.invoke({'messages': [('user', query)]})

In [ ]:
print(result['messages'][-1].content)

아래는 **온디바이스 AI**에 대한 기술·산업·경제·사회 측면의 핵심 내용을 조사해 정리한 것입니다.  
기본적으로 온디바이스 AI는 **클라우드가 아니라 스마트폰·자동차·로봇·가전 등 기기 자체에서 AI 추론을 수행하는 방식**을 뜻합니다.

---

## 1) 기술 동향

### 핵심 기술 축
온디바이스 AI는 크게 3가지 기술 영역으로 발전하고 있습니다.

- **경량화 AI 모델**
  - 프루닝(pruning): 불필요한 파라미터 제거
  - 양자화(quantization): 데이터 해상도 낮춰 연산량 절감
  - 지식 증류 등으로 작은 디바이스에서도 동작 가능한 모델 구현

- **AI 추론 소프트웨어**
  - TensorFlow Lite, Arm NN, NVIDIA TensorRT 같은 추론 프레임워크가 중요
  - 다양한 하드웨어에서 안정적으로 돌아가도록 최적화하는 기술이 핵심

- **전용 AI 반도체(NPU 등)**
  - 저전력·고효율 연산을 위한 NPU, 엣지 TPU, 뉴로모픽 칩 등이 핵심 인프라
  - 국내에서도 사피온, 딥엑스 등 디바이스용/엣지용 NPU 개발이 활발

### 중요한 기술적 특징
- **실시간성**: 네트워크 지연 없이 즉시 반응 가능
- **저전력성**: 모바일·배터리 기기에서 유리
- **프라이버시 보호**: 데이터가 외부 서버로 덜 나가므로 개인정보 보호에 강점
- **한계**:  
  - 학습(training)은 추론보다 훨씬 무거워 온디바이스 환경에서는 어렵고  
  - 범용 환경에서 동작하는 소프트웨어 개발 난도가 높음

---

## 2) 산업 동향

### 글로벌 시장
조사 자료에 따르면 온디바이스 AI 시장은 매우 빠르게 커질 전망입니다.

- **온디바이스 AI 시장**
  - 2024년 약 **271억 달러**
  - 2030년 약 **1,738억 달러**
  - 연평균 성장률(CAGR) **37.7%** 수준 전망

- **생성형 AI 스마트폰 시장**
  - 2027년 약 **5억 2,200

In [ ]:
for event in result['messages']:
    event.pretty_print()

================================ Human Message =================================

온디바이스 AI에 대한 기술, 산업, 경제, 사회 관련 정보를 조사해서 정리해줘
================================== Ai Message ==================================
Tool Calls:
  doc_search (call_EWfIPSADNENVLH0EO724LPVp)
 Call ID: call_EWfIPSADNENVLH0EO724LPVp
  Args:
    query: 온디바이스 AI 기술 산업 경제 사회 동향 정리
================================= Tool Message =================================
Name: doc_search

출처: ./data/온디바이스 AI 기술동향 및 발전방향.pdf
내용: 온디바이스 AI 기술동향 및 발전방향
ISSUE REPORT 2024-06호

출처: ./data/온디바이스 AI 기술동향 및 발전방향.pdf
내용: 온디바이스 AI 기술동향 및 발전방향
ISSUE REPORT 2024-06호

출처: ./data/온디바이스 AI 기술동향 및 발전방향.pdf
내용: DIGISIGHT 2024.06 제6호
Ⅰ. 개 요
Ⅱ. 시장 및 기업 동향 Ⅲ. 핵심기술 동향 Ⅳ. 미래 발전방향 참고문헌 온디바이스 AI 기술동향 및 발전방향

출처: ./data/온디바이스 AI 기술동향 및 발전방향.pdf
내용: DIGISIGHT 2024.06 제6호
Ⅰ. 개 요
Ⅱ. 시장 및 기업 동향 Ⅲ. 핵심기술 동향 Ⅳ. 미래 발전방향 참고문헌 온디바이스 AI 기술동향 및 발전방향
================================== Ai Message ==================================
Tool Calls:
  doc_search (call_3uk48UO

In [ ]:
query = '온디바이스 AI가 소니 플레이스테이션에 어떤 영향을 줄지 알아봐'
result = agentic_rag.invoke({'messages': [('user', query)]})

In [ ]:
for event in result['messages']:
    event.pretty_print()

================================ Human Message =================================

온디바이스 AI가 소니 플레이스테이션에 어떤 영향을 줄지 알아봐
================================== Ai Message ==================================
Tool Calls:
  doc_search (call_DZMuYr49pnOakyYJypbB2wcH)
 Call ID: call_DZMuYr49pnOakyYJypbB2wcH
  Args:
    query: 온디바이스 AI 소니 플레이스테이션 영향
================================= Tool Message =================================
Name: doc_search

출처: ./data/온디바이스 AI 기술동향 및 발전방향.pdf
내용: 온디바이스 AI 기술동향 및 발전방향
ISSUE REPORT 2024-06호

출처: ./data/온디바이스 AI 기술동향 및 발전방향.pdf
내용: 온디바이스 AI 기술동향 및 발전방향
ISSUE REPORT 2024-06호

출처: ./data/온디바이스 AI 기술동향 및 발전방향.pdf
내용: 20 2024.06 제6호
DIGISIGHT
온디바이스 AI 기술동향 및 발전방향
DIGISIGHTESG TREND MEMBER NEWSKEA NOWSTATS
Ⅳ. 미래 발전 방향
1 향후 전망 (시장) 온디바이스 AI 기술은 AI 시장의 중심이 될 것으로 기대
◎ 개인정보 보호, 안정적인 실시간 서비스, 서버 운영비용 절감 등 온디바이스 AI의 다양한 강점으로 인해 향후 AI 시장의 
게임 체인저 역할 예상 - 온디바이스 AI 기술은 대규모 클라우드 서버 운영이 필요 없어 비용 절감 측면에서 혁신적인 기술이며, 나아가 데이터 
센터의 에너지 소모 절감 가능 •  오픈AI는 ’23년 매출이 2조 원이 넘었으나 하루 약 9억 원으

In [ ]:
for event in agentic_rag.stream({'messages': [('user', query)]}):
    print(event)

{'SkillsMiddleware.before_agent': {'skills_metadata': [{'name': 'hyde', 'description': '사용자의 질문과 실제 문서 사이의 표현 차이를 줄이기 위해 질문에 대한 가상의 문서를 생성하고 이를 검색에 활용하는 HyDE 기반 RAG pre-retrieval skill', 'path': '/skills/hyde/SKILL.md', 'metadata': {}, 'license': None, 'compatibility': None, 'allowed_tools': []}, {'name': 'multi-query', 'description': '하나의 질문을 서로 다른 표현과 관점의 여러 검색 질의로 확장하여 다양한 관련 문서를 검색하기 위한 RAG pre-retrieval skill', 'path': '/skills/multi-query/SKILL.md', 'metadata': {}, 'license': None, 'compatibility': None, 'allowed_tools': []}, {'name': 'query-decomposition', 'description': '여러 조건이나 하위 문제를 포함하는 복합 질문을 독립적으로 검색 가능한 하위 질문으로 분해하기 위한 RAG pre-retrieval skill', 'path': '/skills/query-decomposition/SKILL.md', 'metadata': {}, 'license': None, 'compatibility': None, 'allowed_tools': []}, {'name': 'query-rewrite', 'description': '사용자의 질문이 모호하거나 검색에 적합하지 않을 때 원래 의도를 유지하면서 검색에 적합한 질의로 재작성하는 RAG pre-retrieval skill', 'path': '/skills/query-rewrite/SKILL.md', 'metadata': {}, 'license': None, 'co

### Multimodal RAG

In [ ]:
from pathlib import Path

BASE_DIR = Path("multimodal_rag")
IMAGE_DIR = BASE_DIR / "images"

BASE_DIR.mkdir(exist_ok=True)
IMAGE_DIR.mkdir(exist_ok=True)

In [ ]:
# 가상 이미지 생성
from PIL import Image, ImageDraw

img = Image.new("RGB", (800, 500), "white")
draw = ImageDraw.Draw(img)

draw.rounded_rectangle(
    (100, 70, 700, 430),
    radius=30,
    outline="black",
    width=5,
    fill="#eeeeee"
)

draw.text(
    (320, 100),
    "SMART SENSOR X1",
    fill="black"
)

draw.rounded_rectangle(
    (500, 280, 650, 390),
    radius=10,
    outline="black",
    width=4,
    fill="#bbbbbb"
)

draw.text(
    (515, 315),
    "BATTERY",
    fill="black"
)

draw.text(
    (490, 410),
    "Rear Panel Mk33",
    fill="black"
)

rear_path = IMAGE_DIR / "rear_panel.png"
img.save(rear_path)

img = Image.new("RGB", (800, 500), "white")
draw = ImageDraw.Draw(img)

draw.rounded_rectangle(
    (100, 70, 700, 430),
    radius=30,
    outline="black",
    width=5,
    fill="#eeeeee"
)

draw.text(
    (320, 100),
    "SMART SENSOR X1",
    fill="black"
)

draw.ellipse(
    (200, 230, 280, 310),
    outline="black",
    width=4,
    fill="#cccccc"
)

draw.text(
    (205, 325),
    "POWER",
    fill="black"
)

draw.ellipse(
    (500, 240, 540, 280),
    outline="black",
    width=3,
    fill="#dddddd"
)

draw.text(
    (475, 300),
    "STATUS LED",
    fill="black"
)

front_path = IMAGE_DIR / "front_panel.png"
img.save(front_path)

In [ ]:
from langchain_core.documents import Document

# 가상 메뉴얼 생성
documents = [
    Document(
        page_content=(
            "Before replacing the battery, turn off the device "
            "and disconnect the power cable. "
            "Do not replace the battery while the device is operating."
        ),
        metadata={
            "type": "text",
            "section": "Battery Safety"
        }
    ),

    Document(
        page_content=(
            "The Smart Sensor X1 uses a rechargeable lithium-ion battery. "
            "After replacing the battery, close the battery cover securely "
            "before turning the device on."
        ),
        metadata={
            "type": "text",
            "section": "Battery Replacement"
        }
    ),

    Document(
        page_content=(
            "The status LED indicates the operating state of the device. "
            "A continuously blinking LED indicates that the device requires attention."
        ),
        metadata={
            "type": "text",
            "section": "Status LED"
        }
    ),

    Document(
        page_content=(
            "Image of the rear panel of the Smart Sensor X1. "
            "The battery compartment is located in the lower-right area "
            "of the rear panel."
        ),
        metadata={
            "type": "image",
            "image_id": "rear_panel",
            "section": "Device Layout"
        }
    ),

    Document(
        page_content=(
            "Image of the front panel of the Smart Sensor X1. "
            "The power button is located on the left side and "
            "the status LED is located on the right side."
        ),
        metadata={
            "type": "image",
            "image_id": "front_panel",
            "section": "Device Layout"
        }
    ),
]

len(documents)

5

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

db = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    collection_name="multimodal_manual_demo")

In [ ]:
# 텍스트 이미지 검색 도구
@tool
def search_manual(query: str) -> str:
    '''제품 매뉴얼에서 관련된 텍스트 및 이미지 정보를 검색하세요.'''
    docs = db.similarity_search(query)
    results = []

    for i, doc in enumerate(docs, start=1):
        result = [f"[Result {i}]",
                  f"type: {doc.metadata.get('type')}",
                  f"section: {doc.metadata.get('section')}",
                  f"content: {doc.page_content}",]
        if doc.metadata.get("type") == "image":
            result.append(f"image_id: {doc.metadata.get('image_id')}")
        results.append("\n".join(result))
    return "\n\n".join(results)

In [ ]:
# 원본 이미지를 모델에 전달하는 도구
import base64

IMAGE_MAP = {"rear_panel": rear_path, "front_panel": front_path}

@tool
def view_manual_image(image_id: str):
    """image_id를 이용하여 원본 이미지를 전달하는 도구"""

    path = IMAGE_MAP.get(image_id)
    image_base64 = base64.b64encode(path.read_bytes()).decode("utf-8")

    return [
        {"type": "text",
         "text": f"Original manual image: {image_id}"},
        {"type": "image",
         "base64": image_base64,
         "mime_type": "image/png"}]

In [ ]:
system_prompt = """
넌 Smart Sensor X1 매뉴얼에 관한 질문에 답변하는 도우미야.
아래 절차대로 응답해.

# 절차
1. search_manual을 사용하여 관련된 매뉴얼 정보를 검색해.
2. 검색 결과에는 텍스트 문서 또는 이미지 설명이 포함될 수 있어.
3. 이미지 검색 결과에 질문에 답하는 데 필요한 정보가 포함되어 있다면, 해당 image_id를 사용하여 view_manual_image를 호출하고 원본 이미지를 확인해.
4. 검색된 텍스트와 이미지의 정보를 종합해.
5. 매뉴얼에서 확인할 수 있는 정보만을 근거로 답변을 생성해.
6. 답변이 이미지의 시각적 정보에 의존하는 경우, 관련 이미지를 직접 확인하지 않은 상태에서 시각적 정보를 추측하지마.
"""

agent = create_deep_agent(
    model=llm,
    tools=[search_manual, view_manual_image],
    middleware=[TodoListMiddleware()],
    system_prompt=system_prompt)

In [ ]:
query = "배터리는 어디에 설치되어 있으며, 교체하기 전에 무엇을 해야 하나요?"
msg = [("user", query)]
result = agent.invoke({"messages": msg})
result["messages"][-1].pretty_print()

================================== Ai Message ==================================

배터리는 **기기 후면 패널의 오른쪽 아래쪽**에 있는 배터리함에 설치되어 있습니다.

교체하기 전에는 **기기를 끄고 전원 케이블을 분리해야 합니다.** 또한 **기기가 동작 중일 때는 배터리를 교체하면 안 됩니다.**


In [ ]:
for event in result['messages']:
    event.pretty_print()

================================ Human Message =================================

배터리는 어디에 설치되어 있으며, 교체하기 전에 무엇을 해야 하나요?
================================== Ai Message ==================================
Tool Calls:
  search_manual (call_2u2Fm3aAJndw2dXzXkC0KAyT)
 Call ID: call_2u2Fm3aAJndw2dXzXkC0KAyT
  Args:
    query: 배터리는 어디에 설치되어 있으며 교체하기 전에 무엇을 해야 하나요 배터리 설치 위치 교체 전 조치
================================= Tool Message =================================
Name: search_manual

[Result 1]
type: image
section: Device Layout
content: Image of the rear panel of the Smart Sensor X1. The battery compartment is located in the lower-right area of the rear panel.
image_id: rear_panel

[Result 2]
type: text
section: Battery Safety
content: Before replacing the battery, turn off the device and disconnect the power cable. Do not replace the battery while the device is operating.

[Result 3]
type: text
section: Battery Replacement
content: The Smart Sensor X1 uses a rechargeable lithium-ion battery. Af